In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

In [ ]:
# DATA PREPARATION FOR ABLATION STUDY
import pandas as pd
import numpy as np
df = pd.read_csv("C:\\Users\\delta\\Predictive-maintenence-iot\\dataset\\ai4i2020.csv")
print("Dataset Loaded Successfully")
print("Dataset Shape:", df.shape)
# Target Variable
target = "Machine failure"
y = df[target]
print("\nTarget Variable Distribution")
print(y.value_counts())
# Baseline Dataset and Sensor Features Only
baseline_features = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]'
]
X_baseline = df[baseline_features]
print("\nBaseline Dataset Created")
print("Features Used:")
for feature in baseline_features:
    print("-", feature)
print("\nBaseline Dataset Shape:")
print(X_baseline.shape)
# Context Dataset and Sensor + Context Features
context_features = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]',
    'Type'
]
X_context = df[context_features]
# Convert categorical Machine Type
# into numerical columns
X_context = pd.get_dummies(
    X_context,
    columns=['Type'],
    drop_first=True
)
print("\nContext Dataset Created")
print("Features Used:")
for feature in context_features:
    print("-", feature)
print("\nContext Dataset Shape:")
print(X_context.shape)
# Dataset Summary
summary = pd.DataFrame({
    "Dataset": [
        "Baseline Dataset",
        "Context Dataset"
    ],
    "Description": [
        "Sensor Features Only",
        "Sensor + Machine Type"
    ],
    "Number of Features": [
        X_baseline.shape[1],
        X_context.shape[1]
    ]
})
print("\nDataset Summary")
display(summary)
# Preview datasets
print("\nBaseline Dataset Preview")
display(X_baseline.head())
print("\nContext Dataset Preview")
display(X_context.head())

In [ ]:
# Baseline Split
Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    X_baseline,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
# Context Split
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_context,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print("Datasets split successfully.")

In [ ]:
baseline_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)
baseline_model.fit(
    Xb_train,
    yb_train
)
print("Baseline model trained.")

In [ ]:
context_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)
context_model.fit(
    Xc_train,
    yc_train
)
print("Context model trained.")

In [ ]:
context_model = LogisticRegression( max_iter=1000, random_state=42)
context_model.fit(Xc_train,yc_train)
print("Context model trained.")

In [ ]:
baseline_pred = baseline_model.predict(Xb_test)
context_pred = context_model.predict(Xc_test)
baseline_prob = baseline_model.predict_proba(Xb_test)[:,1]
context_prob = context_model.predict_proba(Xc_test)[:,1]
print("Predictions generated.")

In [ ]:
def calculate_metrics(y_true, y_pred, y_prob):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1 Score": f1_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_prob)
    }

In [ ]:
baseline_results = calculate_metrics( yb_test, baseline_pred, baseline_prob)
context_results = calculate_metrics( yc_test, context_pred, context_prob)
print("Evaluation completed.")

In [ ]:
comparison = pd.DataFrame({
    "Metric": baseline_results.keys(),
    "Baseline": baseline_results.values(),
    "Context": context_results.values()
})
comparison["Improvement"] = (
    comparison["Context"] -
    comparison["Baseline"]
)
comparison

In [ ]:
best_metric = comparison.loc[
    comparison["Improvement"].idxmax()
]
print("Highest Improvement")
print("------------------")
print("Metric:", best_metric["Metric"])
print("Gain:", round(best_metric["Improvement"],4))

In [ ]:
plt.figure(figsize=(10,5))
x = np.arange(len(comparison))
width = 0.35
plt.bar( x-width/2, comparison["Baseline"], width, label="Baseline")
plt.bar( x+width/2, comparison["Context"], width, label="Context")
plt.xticks( x, comparison["Metric"])
plt.ylabel("Score")
plt.title( "Baseline vs Context Model Performance")
plt.legend()
plt.show()
plt.savefig("Baseline_vs_Context_Model_Performance.png")

In [ ]:
plt.figure(figsize=(8,5))
plt.bar(comparison["Metric"], comparison["Improvement"])
plt.title("Improvement from Context Features")
plt.ylabel("Metric Gain")
plt.show()
plt.savefig("Improvement_from_Context_Features.png")

In [ ]:
cm_baseline = confusion_matrix( yb_test, baseline_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_baseline)
disp.plot()
plt.title("Baseline Model Confusion Matrix")
plt.show()
plt.savefig("baseline_confusion_matrix.png")

In [ ]:
cm_context = confusion_matrix( yc_test, context_pred)
disp = ConfusionMatrixDisplay( confusion_matrix=cm_context)
disp.plot()
plt.title("Context Model Confusion Matrix")
plt.show()
plt.savefig("context_model_confusion_matrix.png")

In [ ]:
avg_baseline = comparison["Baseline"].mean()
avg_context = comparison["Context"].mean()
overall_gain = avg_context - avg_baseline
print("Average Baseline Score :", round(avg_baseline,4))
print("Average Context Score  :", round(avg_context,4))
print("Overall Improvement    :", round(overall_gain,4))

In [ ]:
report = f"""
ABLATION STUDY RESULTS
================================================

Models Compared
---------------
1. Baseline Model
2. Context Model

Metrics Evaluated
-----------------
Accuracy
Precision
Recall
F1 Score
ROC-AUC

Average Baseline Score : {avg_baseline:.4f}
Average Context Score  : {avg_context:.4f}

Overall Improvement    : {overall_gain:.4f}

Conclusion
----------
The Context Model was evaluated against the
Baseline Model to determine the contribution
of contextual features.

Positive metric improvements indicate that
additional context information helps improve
machine failure prediction.

================================================
"""

print(report)